In [0]:
import requests
from datetime import date

ENDPOINT = "https://archive-api.open-meteo.com/v1/archive"
LATITUDE = 40.7128
LONGITUDE = -74.0060
HOURLY_VARS = "temperature_2m,precipitation,weather_code"
START_DATE = "2026-03-01"
END_DATE = "2026-05-31"

# Proposed model pin

WEATHER_MODEL = "era5"

params = {
    "latitude": LATITUDE,
    "longitude": LONGITUDE,
    "start_date": START_DATE,
    "end_date": END_DATE,
    "hourly": HOURLY_VARS,
    "timezone": "UTC",
    "models": WEATHER_MODEL,
}

response = requests.get(ENDPOINT, params=params, timeout=30)
response.raise_for_status()  # status-code check -- but HTTP 200 alone is not treated as success below
payload = response.json()

# --- response structure / schema, printed explicitly so it's actually evidence ---
print("status_code:", response.status_code)
print("top-level keys:", list(payload.keys()))
print("hourly keys:", list(payload["hourly"].keys()))
print("hourly_units:", payload.get("hourly_units"))
print("returned latitude/longitude:", payload.get("latitude"), payload.get("longitude"))
print("elevation:", payload.get("elevation"))
print("utc_offset_seconds:", payload.get("utc_offset_seconds"))
print("timezone / timezone_abbreviation:", payload.get("timezone"), payload.get("timezone_abbreviation"))
print("generationtime_ms:", payload.get("generationtime_ms"))
print("pinned model requested:", WEATHER_MODEL)

# --- HARD validation: "HTTP 200 alone is not treated as success" ---

# 1. Response structure: required top-level fields must be present.
REQUIRED_TOP_LEVEL_FIELDS = (
    "latitude", "longitude", "elevation", "utc_offset_seconds",
    "timezone", "timezone_abbreviation", "hourly", "hourly_units",
)
missing_top_level = [f for f in REQUIRED_TOP_LEVEL_FIELDS if f not in payload]
if missing_top_level:
    raise ValueError(f"Response is missing required top-level field(s): {missing_top_level}")

hourly = payload["hourly"]

# 2. Expected fields / non-empty payload, and variable support for the
# pinned model (per docs/ingestion.md: "select and pin a model after
# confirming variable support").
for var_name in ("time", "temperature_2m", "precipitation", "weather_code"):
    values = hourly.get(var_name)
    if not values:
        raise ValueError(
            f"Pinned model '{WEATHER_MODEL}' returned no '{var_name}' values -- "
            "non-empty-payload check failed."
        )

for var_name in ("temperature_2m", "precipitation", "weather_code"):
    null_count = sum(1 for v in hourly[var_name] if v is None)
    print(f"{var_name}: {len(hourly[var_name])} values, {null_count} null")
    if null_count > 0:
        print(
            f"  WARNING: '{WEATHER_MODEL}' returned nulls for '{var_name}' -- "
            "this model may not fully support this variable at this location/window."
        )

# 3. Date coverage: a HARD check, not just an informational print -- a
# coverage mismatch fails the ingestion instead of silently landing a
# partial window as if it were complete.
expected_hours = ((date.fromisoformat(END_DATE) - date.fromisoformat(START_DATE)).days + 1) * 24
actual_hours = len(hourly["time"])
print("expected_hours:", expected_hours, "| actual_hours:", actual_hours)
if actual_hours != expected_hours:
    raise ValueError(
        f"Date coverage check failed: expected {expected_hours} hourly rows for "
        f"{START_DATE}..{END_DATE}, got {actual_hours}. Refusing to land a partial "
        "window as if it were complete."
    )

weather_rows = list(zip(hourly["time"], hourly["temperature_2m"], hourly["precipitation"], hourly["weather_code"]))

weather_df = spark.createDataFrame(weather_rows, ["time", "temperature_2m", "precipitation", "weather_code"])

display(weather_df)


In [0]:
# Save the raw response as immutable, versioned evidence for issue #11

import json, hashlib, os
from datetime import datetime, timezone as dt_timezone

LANDING_DIR = "/Volumes/ftw-week-08/00-source/group_a_source/weather"
os.makedirs(LANDING_DIR, exist_ok=True)

# Fixed filename for bronze SQL to read (one-time historical load)
RAW_FILENAME = "open_meteo_mar_may_2026.json"
RAW_PATH = os.path.join(LANDING_DIR, RAW_FILENAME)

with open(RAW_PATH, "w") as f:
    json.dump(payload, f)

checksum = hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()
print("saved raw response to:", RAW_PATH)
print("sha256:", checksum)
print("paste this file's contents (or the checksum + path) into issue #11 as the committed sample response")

# Capture response-level metadata

response_metadata_row = {
    "source_system": "open_meteo",
    "source_url": ENDPOINT,
    "source_file": RAW_PATH,
    "source_file_version": checksum,
    "weather_model": WEATHER_MODEL,
    "requested_latitude": LATITUDE,
    "requested_longitude": LONGITUDE,
    "requested_start_date": START_DATE,
    "requested_end_date": END_DATE,
    "returned_latitude": payload.get("latitude"),
    "returned_longitude": payload.get("longitude"),
    "elevation_m": payload.get("elevation"),
    "utc_offset_seconds": payload.get("utc_offset_seconds"),
    "timezone": payload.get("timezone"),
    "timezone_abbreviation": payload.get("timezone_abbreviation"),
}
print("response_metadata_row (not yet persisted to a control table):", response_metadata_row)


In [0]:
weather_df.count()

In [0]:
weather_df.createOrReplaceTempView("weather_hourly")

In [0]:
%sql
SELECT
    time,
    COUNT(*) AS duplicate_count
FROM weather_hourly
GROUP BY time
HAVING COUNT(*) > 1;

In [0]:
%sql
SELECT
    SUM(CASE WHEN time IS NULL THEN 1 ELSE 0 END) AS null_time,
    SUM(CASE WHEN temperature_2m IS NULL THEN 1 ELSE 0 END) AS null_temperature_2m,
    SUM(CASE WHEN precipitation IS NULL THEN 1 ELSE 0 END) AS null_precipitation,
    SUM(CASE WHEN weather_code IS NULL THEN 1 ELSE 0 END) AS null_weather_code
FROM weather_hourly;

In [0]:
%sql
SELECT DISTINCT weather_code
FROM weather_hourly
ORDER BY weather_code;

In [0]:
%sql
SELECT *
FROM weather_hourly
WHERE precipitation < 0
   OR temperature_2m < -50
   OR temperature_2m > 60;

In [0]:
from datetime import date

expected_hours = ((date(2026, 5, 31) - date(2026, 3, 1)).days + 1) * 24
actual_hours = weather_df.count()

print("expected_hours:", expected_hours)
print("actual_hours:", actual_hours)
print("gap:", expected_hours - actual_hours)

In [0]:
# Empty-response / error behaviour probe (out-of-range window, not part of the trusted dataset)
oor_params = dict(params, start_date="2099-01-01", end_date="2099-01-02")
oor_response = requests.get(ENDPOINT, params=oor_params, timeout=30)

print("status_code:", oor_response.status_code)
print("body:", oor_response.text[:500])

In [0]:
# Repeated identical request -- determinism check (drop generationtime_ms, which varies per call)
repeat_response = requests.get(ENDPOINT, params=params, timeout=30)
repeat_payload = repeat_response.json()

def normalize(body):
    b = dict(body)
    b.pop("generationtime_ms", None)
    return b

print("repeat status_code:", repeat_response.status_code)
print("same content on repeat:", normalize(payload) == normalize(repeat_payload))

In [0]:
# Rate-limit behaviour
# Open-Meteo's non-commercial free tier needs no API key and returns no
# per-request rate-limit headers (confirmed below -- scan for any). Its
# documented limits are daily/monthly call quotas published at
# https://open-meteo.com/en/docs/historical-weather-api -- record the actual
# quota figures from that page in the write-up. The out-of-range probe above
# already exercises the non-200 path a bounded retry (429/5xx + backoff)
# should catch in real ingestion code.
print("response headers (scan for any rate-limit hints):")
for k, v in response.headers.items():
    print(f"  {k}: {v}")